# Demo: phân loại nguyên nhân lỗi build CI

Cần: Runtime GPU (T4 là đủ), secret `KAGGLE_API_TOKEN` và `NGROK_AUTH_TOKEN` (biểu tượng chìa khoá). Chạy lần lượt các ô; ô 3 in link ngrok để mở demo. Giữ tab mở trong lúc demo; xong thì Runtime → Disconnect and delete runtime để không tốn compute unit.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
import os, sys, glob, subprocess, zipfile
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
subprocess.run("pip install -q -U kaggle gradio pyngrok duckdb", shell=True)
for ds, d in [("p3-demo", "/content/p3demo"), ("p3-c-weights", "/content/p3w"), ("p3-gh-token", "/content/p3tok"),
              ("p3-cause-weights", "/content/p3cw")]:
    r = subprocess.run(f"kaggle datasets download ducanhpham2k5/{ds} -p {d} --unzip --force", shell=True,
                       capture_output=True, text=True)
    print(ds, r.stdout[-120:], r.stderr[-200:])
for z in glob.glob("/content/p3demo/*.zip"):  # Kaggle thường tự giải nén; nếu chưa thì giải nén ở đây
    zipfile.ZipFile(z).extractall("/content/p3demo/" + os.path.basename(z)[:-4])
sys.path.insert(0, "/content/p3demo/p3demo_code/src")
os.environ["GITHUB_TOKEN"] = open(glob.glob("/content/p3tok/**/gh_token.txt", recursive=True)[0]).read().strip()
print(sorted(os.listdir("/content/p3demo")), sorted(os.listdir("/content/p3w")))

In [ ]:
from project3.demo.app import launch
url = launch("/content/p3demo", "/content/p3w/C_s0_fold{fold}.model_soft0.pt", ngrok_token=userdata.get("NGROK_AUTH_TOKEN"),
             cause_weights="/content/p3cw/CAUSED_s0_fold{fold}.model_soft0.pt")  # dòng nguyên nhân (16/11)
print("Mở demo tại:", url)

In [ ]:
# kiểm tra nhanh: dự đoán 6 run mẫu (không qua giao diện)
from project3.demo.app import load_examples
from project3.demo.predict import Predictor
p = Predictor("/content/p3demo", "/content/p3w/C_s0_fold{fold}.model_soft0.pt",
              cause_weights="/content/p3cw/CAUSED_s0_fold{fold}.model_soft0.pt")
for rid, (note, rec) in load_examples("/content/p3demo").items():
    o = p.predict(rec)
    print(rid, note, "->", o["pred"]["leaf"], f'{o["pred"]["p_leaf"]:.3f}', "| entropy", round(o["mc"]["entropy"], 2),
          "| trả lời" if o["mc"]["answered"] else "| không chắc", "| dòng 1:", o["lines"][0]["text"][:80], "| nguyên nhân 1:", o["cause_lines"][0]["text"][:80], f'({o["seconds"]}s)')